# DataCleaningEnv — OpenEnv Hands-On Notebook

**Course:** Meta × Scaler OpenEnv Hackathon  
**Interface:** OpenEnv 3-method spec — `reset()` · `step()` · `state()`  

This notebook walks you through the full lifecycle of `DataCleaningEnv` — a reinforcement-learning environment where an agent learns to clean dirty tabular data.

**What you'll do:**
1. Install dependencies and import the environment
2. Explore the action space and observation format
3. Run a single episode manually, step by step
4. Use the `state()` method to inspect full environment state
5. Run the baseline agent across all difficulty levels
6. Connect via the typed client to the live server
7. Challenge: write your own agent and beat the baseline


## 1 · Setup

Install dependencies and import everything you need.

In [ ]:
# Install dependencies (run once)
# If running locally, skip this and install via: pip install -r requirements.txt
%pip install -q pandas numpy fastapi uvicorn httpx pydantic>=2.0

In [ ]:
import pandas as pd
import numpy as np

from env import DataCleaningEnv
from agent import baseline_agent
from models import (
    ActionModel,
    EnvStateModel,
    ObservationModel,
    StepResultModel,
    VALID_ACTIONS,
)

print('All imports OK')


## 2 · Action Space

Every OpenEnv environment defines a fixed set of actions the agent can take. `DataCleaningEnv` has three:

| ID | Name | When to use |
|---|---|---|
| `0` | `skip` | Do nothing — move to the next issue |
| `1` | `impute_missing` | Fill a `NaN` cell with the column mean/mode |
| `3` | `fix_outlier` | Replace an outlier/invalid value with the column mean |

> **Note:** Action `2` is intentionally absent — it is reserved for a future `flag_for_review` action and intentionally left unassigned to keep existing agent checkpoints compatible.

All three actions are validated by `ActionModel` from `models.py`.

In [ ]:
# Inspect the valid action set
print('Valid actions:', VALID_ACTIONS)

# ActionModel validates before anything hits the environment
valid = ActionModel(action=1)
print('Valid action label:', valid.label)

try:
    ActionModel(action=2)   # action 2 does not exist
except Exception as e:
    print('Invalid action correctly rejected:', type(e).__name__)


## 3 · Reset and First Observation

`reset(task_level)` starts a new episode and returns the first observation — a dict with a single key `row_data` containing the first problematic row.

Task levels: `'easy'` (4 rows) · `'medium'` (5 rows) · `'hard'` (8 rows)

In [ ]:
env = DataCleaningEnv()
obs = env.reset(task_level='medium')

print('First observation:')
print(obs)
print()
print('row_data:', obs['row_data'])
print()
# Validate it matches the typed model
# (sanitise NaN -> None first for Pydantic)
row_clean = {k: (None if (isinstance(v, float) and np.isnan(v)) else v)
             for k, v in obs['row_data'].items()}
obs_model = ObservationModel(row_data=row_clean)
print('ObservationModel:', obs_model)


## 4 · state() — Full Environment Snapshot

`state()` is the third required OpenEnv method. It returns a complete snapshot of the episode without advancing it — useful for logging, debugging, and building training loops.

The returned dict maps directly onto `EnvStateModel` in `models.py`.

In [ ]:
s = env.state()

print(f"Task level      : {s['task_level']}")
print(f"Step            : {s['current_step']} / {s['max_steps']}")
print(f"Issues at start : {s['total_issues_at_start']}")
print(f"Remaining issues: {s['remaining_issues']}")
print(f"Score so far    : {s['score']}")
print(f"Done            : {s['done']}")
print(f"Observation     : {s['current_observation']}")
print(f"Episode log     : {s['episode_log']}  (empty — no steps yet)")


## 5 · Manual Step-by-Step Episode

`step(action)` returns `(observation, reward, done, info)`.

Reward signal:
- `+2` correct fix
- `−1` wrong action
- `+5` bonus when all issues resolved
- `−5` penalty if max steps reached before completion

In [ ]:
env2 = DataCleaningEnv()
obs2 = env2.reset(task_level='easy')

print('=== Easy episode — step by step ===\n')
step_num = 0
done = False

while not done:
    step_num += 1
    row = obs2['row_data']

    # Decide action using the baseline agent
    action = baseline_agent(obs2)
    action_name = VALID_ACTIONS.get(action, str(action))

    obs2, reward, done, _ = env2.step(action)

    print(f'Step {step_num}: action={action} ({action_name:20s})  '
          f'reward={reward:+d}  done={done}')

print()
print(f'Final score : {env2.grade()}')
print(f'Total steps : {env2.steps}')


In [ ]:
# Inspect the full episode log after completion
log_df = pd.DataFrame(env2.episode_log)
log_df['correct'] = log_df['correct'].map({True: '✅', False: '❌'})
log_df['reward']  = log_df['reward'].apply(lambda r: f'{r:+.0f}')
print(log_df.to_string(index=False))


## 6 · state() After Episode Completion

Once `done=True`, `state()` reflects the final snapshot: `score=1.0`, `current_observation=None`, `done=True`.

In [ ]:
final_state = env2.state()
print('done :', final_state['done'])
print('score:', final_state['score'])
print('obs  :', final_state['current_observation'])  # None when done
print('log  :', len(final_state['episode_log']), 'entries')


## 7 · Baseline Agent — All Difficulty Levels

Run the rule-based `baseline_agent` across easy / medium / hard and compare scores.

In [ ]:
results = []

for level in ['easy', 'medium', 'hard']:
    env = DataCleaningEnv()
    env._task_level = level
    obs = env.reset(level)
    done = False
    total_reward = 0

    while not done:
        action = baseline_agent(obs)
        obs, reward, done, _ = env.step(action)
        total_reward += reward

    results.append({
        'task'  : level,
        'score' : env.grade(),
        'reward': total_reward,
        'steps' : env.steps,
        'issues': env.total_issues_at_start,
    })

print(f"{'Task':<8} {'Score':>6} {'Reward':>8} {'Steps':>6} {'Issues':>7}")
print('-' * 40)
for r in results:
    print(f"{r['task']:<8} {r['score']:>6.2f} {r['reward']:>8} {r['steps']:>6} {r['issues']:>7}")


## 8 · Running via the OpenEnv Server + Typed Client

The environment can also be served as a FastAPI HTTP server and accessed through the typed `DataCleaningClient`. This is the production OpenEnv pattern.

**Terminal 1 — start the server:**
```bash
python app.py
# or: uvicorn app:app --host 0.0.0.0 --port 8000 --reload
```

**Terminal 2 (or this notebook) — connect with the client:**

In [ ]:
# Make sure the server is running before executing this cell.
# Start it with: python app.py

from client import DataCleaningClient

with DataCleaningClient(base_url='http://localhost:8000').sync() as client:

    # reset() — start a new episode
    obs = client.reset(task_level='hard')
    print('reset() →', obs)

    # state() — full snapshot before any steps
    s = client.state()
    print(f'state()  → task={s.task_level}  issues={s.total_issues_at_start}  score={s.score}')

    # Run the full episode via the client
    done = False
    total_reward = 0
    while not done:
        action = baseline_agent({'row_data': obs.row_data} if obs else None)
        result = client.step(action=action)
        total_reward += result.reward.value
        done = result.reward.done
        obs = result.observation

    # Final state
    final = client.state()
    print(f'done  → score={final.score}  steps={final.current_step}  reward={total_reward}')


## 9 · Challenge — Write Your Own Agent

The baseline agent uses simple if/else rules. Can you do better?

Your agent receives one observation at a time:
```python
obs = {
    'row_data': {
        'age': None,        # None = missing
        'salary': 1000000,  # > 200000 = outlier
        'city': 'NY',
        'experience': 3.0,
        'rating': 4.2,
    }
}
```
And must return one of: `0` (skip) · `1` (impute_missing) · `3` (fix_outlier)

**Ideas to try:**
- A look-ahead agent that checks all values in the row before deciding
- An LLM agent that sends the observation to Claude / GPT and parses the response
- A priority-queue agent that fixes outliers before missing values

In [ ]:
def my_agent(observation):
    """
    TODO: write your own agent here.

    Parameters
    ----------
    observation : dict | None
        {'row_data': {'age': ..., 'salary': ..., 'city': ...,
                      'experience': ..., 'rating': ...}}
        None when the episode is already done.

    Returns
    -------
    int : 0 (skip) | 1 (impute_missing) | 3 (fix_outlier)
    """
    if observation is None:
        return 0

    row = observation['row_data']

    # --- your logic here ---
    return baseline_agent(observation)   # replace with your own strategy


# Benchmark your agent against the baseline
def benchmark(agent_fn, task_level='hard'):
    env = DataCleaningEnv()
    obs = env.reset(task_level)
    done = False; total_reward = 0
    while not done:
        action = agent_fn(obs)
        obs, reward, done, _ = env.step(action)
        total_reward += reward
    return {'score': env.grade(), 'reward': total_reward, 'steps': env.steps}


print('Baseline  :', benchmark(baseline_agent, 'hard'))
print('Your agent:', benchmark(my_agent,       'hard'))
